                                User–Item Rating Matrix

In [ ]:
import numpy as np

# -------------------------------
# 1. Define User–Item Matrix
# -------------------------------
ratings = np.array([
    [5, 3, 0, 1],
    [4, 0, 0, 1],
    [1, 1, 0, 5],
    [0, 0, 5, 4]
], dtype=float)

num_users, num_items = ratings.shape
print("User–Item Rating Matrix:\n", ratings)


User–Item Rating Matrix:
 [[5. 3. 0. 1.]
 [4. 0. 0. 1.]
 [1. 1. 0. 5.]
 [0. 0. 5. 4.]]


In [2]:
# -------------------------------
# 2. Cosine Similarity Function
# -------------------------------
def cosine_similarity(matrix):
    dot_product = matrix @ matrix.T
    norms = np.linalg.norm(matrix, axis=1)
    similarity = dot_product / (norms[:, None] * norms[None, :] + 1e-8)
    return similarity

user_similarity = cosine_similarity(ratings)
print("\nUser Similarity Matrix:\n", np.round(user_similarity, 3))


User Similarity Matrix:
 [[1.    0.861 0.423 0.106]
 [0.861 1.    0.42  0.152]
 [0.423 0.42  1.    0.601]
 [0.106 0.152 0.601 1.   ]]


In [3]:
# -------------------------------
# 3. Predict Missing Ratings
#    Using Weighted Sum of Neighbors
# -------------------------------
def predict_ratings(ratings, similarity):
    predicted = np.zeros(ratings.shape)
    
    for u in range(num_users):
        for i in range(num_items):
            if ratings[u, i] == 0:  # Only predict missing ratings
                sim_scores = similarity[u]
                item_ratings = ratings[:, i]

                mask = item_ratings > 0
                if np.sum(mask) == 0:
                    predicted[u, i] = 0
                else:
                    predicted[u, i] = np.sum(sim_scores[mask] * item_ratings[mask]) / np.sum(sim_scores[mask])

            else:
                predicted[u, i] = ratings[u, i]
    
    return predicted

predicted_matrix = predict_ratings(ratings, user_similarity)
print("\nPredicted Rating Matrix:\n", np.round(predicted_matrix, 2))




Predicted Rating Matrix:
 [[5.   3.   5.   1.  ]
 [4.   2.34 5.   1.  ]
 [1.   1.   5.   5.  ]
 [2.02 1.3  5.   4.  ]]


In [4]:
# -------------------------------
# 4. Recommend Top-N Items
# -------------------------------
def recommend_items(pred_matrix, user, n=2):
    user_ratings = pred_matrix[user]
    original = ratings[user]
    
    unrated_idx = np.where(original == 0)[0]
    unrated_predictions = user_ratings[unrated_idx]
    
    top_indices = unrated_idx[np.argsort(unrated_predictions)[::-1][:n]]
    return top_indices, unrated_predictions[np.argsort(unrated_predictions)[::-1][:n]]

user = 0
items, scores = recommend_items(predicted_matrix, user, n=2)
print(f"\nTop recommended items for User {user+1}:")
for item, score in zip(items, scores):
    print(f"  Item {item+1} (predicted rating: {score:.2f})")



Top recommended items for User 1:
  Item 3 (predicted rating: 5.00)
